# 🧬 FIAP Tech Challenge - Fase 2 (Grupo 40)

**Integrantes:** Felipe Antonio Ayancan Hernandez e Fernando Ponz Vargas  
**Projeto Selecionado:** Projeto 1 - Otimização de Modelos de Diagnóstico Oncológico  

---

## 🎯 Contextualização e Conexão com a Fase 1

Na **Fase 1**, construímos e avaliamos modelos preditivos para o diagnóstico precoce de tumores de mama (benignos vs. malignos) utilizando o dataset *Breast Cancer Wisconsin*. O trabalho atingiu **100% de aproveitamento**, com destaques no rigor analítico e explicabilidade com SHAP.

### 💡 Feedback do Professor & Oportunidade Técnica:
> *"Experimente extrair as probabilidades (predict_proba) da sua Regressão Logística otimizada e reduzir o limiar de decisão de 0,5 para 0,3 para verificar se os 2 falsos negativos restantes no teste zeram, aumentando o colchão de segurança da triagem hospitalar."*

Na **Fase 2**, elevamos essa diretriz para o estado da arte:
1. **Algoritmos Genéticos (DEAP):** Em vez de tunar parâmetros manualmente, modelamos um **cromossomo híbrido** contendo hiperparâmetros matemáticos ($C$, regularização, ponderação de classes) e o próprio **Limiar Clínico (Threshold)** como um gene contínuo. Realizamos **3 experimentos completos** com diferentes configurações genéticas.
2. **Fitness Function Clínica:** Focada na maximização do *Recall* (segurança do paciente) combinada com *F1-Score*.
3. **Explicabilidade com IA Generativa (LLM) + SHAP:** O modelo alimenta uma LLM com as probabilidades e os vetores de SHAP para gerar **laudos oncológicos estruturados em linguagem natural**.
4. **Aplicação Interativa:** Sistema web em Streamlit com inferência em tempo real.

In [ ]:
import os
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from src.model_trainer import load_and_prep_data, build_logistic_pipeline
from src.genetic_optimizer import run_all_experiments
from src.llm_explainer import generate_medical_report
from sklearn.metrics import confusion_matrix, classification_report, roc_curve, auc

## 1. Carregamento dos Dados da Fase 1

In [ ]:
X_train, X_test, y_train, y_test = load_and_prep_data('data/data.csv')
print(f"Conjunto de Treino: {X_train.shape[0]} amostras, {X_train.shape[1]} features")
print(f"Conjunto de Teste:  {X_test.shape[0]} amostras, {X_test.shape[1]} features")
print(f"Distribuição no Teste -> Benignos: {(y_test == 0).sum()} | Malignos: {(y_test == 1).sum()}")

## 2. Execução dos 3 Experimentos com Algoritmo Genético

Configurações testadas conforme o edital da Fase 2:
- **Experimento 1 (Conservador):** População = 20, 10 gerações, Crossover = 50%, Mutação = 20%
- **Experimento 2 (Alta Exploração):** População = 35, 15 gerações, Crossover = 70%, Mutação = 40%
- **Experimento 3 (População Expandida):** População = 50, 20 gerações, Crossover = 60%, Mutação = 25%

In [ ]:
# Carregamos os resultados gerados pela suíte de Algoritmos Genéticos (src.genetic_optimizer)
exp_data = joblib.load('data/experiments_results.joblib')
experiments = exp_data['experiments']
comp_df = exp_data['comparison_df']

for exp in experiments:
    ind = exp['best_individual']
    print(f"{exp['exp_name']}:")
    print(f"  -> Fitness: {exp['best_fitness']:.4f} | C: {ind[0]:.4f} | CW: {ind[1]} | Threshold Clínico: {ind[2]:.4f}")

### 📈 Curvas de Convergência dos 3 Experimentos Genéticos

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharey=True)
colors = ['#1f77b4', '#ff7f0e', '#2ca02c']

for i, exp in enumerate(experiments):
    logbook = exp['logbook']
    gen = [entry['gen'] for entry in logbook]
    avg_f = [entry['avg'] for entry in logbook]
    max_f = [entry['max'] for entry in logbook]
    
    ax = axes[i]
    ax.plot(gen, avg_f, label='Média da População', linestyle='--', color='gray')
    ax.plot(gen, max_f, label='Melhor Indivíduo', color=colors[i], lw=2.5)
    ax.set_title(exp['exp_name'], fontsize=11, fontweight='bold')
    ax.set_xlabel('Geração')
    if i == 0:
        ax.set_ylabel('Fitness Composta (Recall + F1)')
    ax.grid(True, alpha=0.3)
    ax.legend(loc='lower right')

plt.suptitle('Convergência Evolutiva do Algoritmo Genético nas Configurações Avaliadas', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Comparativo de Desempenho no Teste Cego: Fase 1 vs. Fase 2

Abaixo comparamos o modelo baseline da Fase 1 ($T=0.50$) com o modelo otimizado pelo Algoritmo Genético ($T=0.1815$).

In [ ]:
print(comp_df.to_string(index=False))

# Matrizes de Confusão Lado a Lado
cm_base = exp_data['cm_base']
cm_opt = exp_data['cm_opt']

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

sns.heatmap(cm_base, annot=True, fmt='d', cmap='Blues', cbar=False, ax=ax1,
            xticklabels=['Benigno', 'Maligno'], yticklabels=['Benigno', 'Maligno'])
ax1.set_title('Fase 1: Baseline (Threshold = 0.50)\n3 Falsos Negativos', fontweight='bold')
ax1.set_xlabel('Predição')
ax1.set_ylabel('Real')

sns.heatmap(cm_opt, annot=True, fmt='d', cmap='Greens', cbar=False, ax=ax2,
            xticklabels=['Benigno', 'Maligno'], yticklabels=['Benigno', 'Maligno'])
ax2.set_title('Fase 2: AG + Threshold Clínico (Threshold = 0.18)\n1 Falso Negativo (Recall 97.6%)', fontweight='bold')
ax2.set_xlabel('Predição')
ax2.set_ylabel('Real')

plt.tight_layout()
plt.show()

## 4. Integração com LLM e Explicabilidade Clínica (SHAP)

Demonstração da geração de laudos em linguagem natural para profissionais de saúde a partir das métricas calculadas pelo modelo otimizado.

In [ ]:
best_model_art = joblib.load('data/best_model_phase2.joblib')
pipeline = best_model_art['pipeline']
threshold = best_model_art['best_threshold']
feature_names = best_model_art['feature_names']

# Seleção de um caso crítico de teste
critical_idx = y_test[y_test == 1].index[0]
sample = X_test.loc[[critical_idx]]
prob = float(pipeline.predict_proba(sample)[0, 1])
is_malignant = prob >= threshold

# Extração SHAP linear local
scaler = pipeline.named_steps['scaler']
model = pipeline.named_steps['model']
scaled = scaler.transform(sample)
linear_imp = scaled[0] * model.coef_[0]
top_shap = sorted(zip(feature_names, linear_imp), key=lambda x: abs(x[1]), reverse=True)[:4]

print(f"Paciente Teste #{critical_idx}:")
print(f"Probabilidade Preditiva: {prob*100:.2f}%")
print(f"Threshold Clinico Otimizado: {threshold*100:.2f}%")
print(f"Classificacao: {'Maligno (Alto Risco)' if is_malignant else 'Benigno (Baixo Risco)'}")
print("Top Biomarcadores SHAP:", top_shap)

In [ ]:
# Geração do Laudo Médico Clínico
laudo = generate_medical_report(
    patient_features=sample[['radius_mean', 'perimeter_mean', 'concavity_mean', 'area_mean']].to_dict(orient='records')[0],
    prediction_proba=prob,
    clinical_threshold=threshold,
    is_malignant=is_malignant,
    shap_top_features=top_shap
)
print(laudo)

## 5. Conclusão da Fase 2

- **Atendimento ao Feedback da Fase 1:** O limiar clínico foi integrado à busca evolutiva, provando matematicamente a redução drástica de falsos negativos (Recall saltando de 92.86% para 97.62%).
- **Evolução Genética Eficiente:** O Experimento 2 (com maior diversidade genética) alcançou o melhor compromisso clínico.
- **Explicabilidade Pronta para Uso:** A integração do SHAP com o motor de linguagem natural permite que médicos compreendam o racional da decisão sem atrito matemático.
- **Próximos Passos (Fase 3):** Incorporação de dados textuais de prontuários e consolidação de um assistente médico multimodal autônomo.